# Forward Curve Modeling Example

This notebook demonstrates how to use the forward curve modeling infrastructure with OKX orderbook data.

We'll cover:
1. Pillar preparation
2. PCHIP interpolation with EWMA smoothing
3. Kalman-filtered Nelson-Siegel carry model
4. Reconstructing forwards at specific (unlisted) tenors


In [22]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, timedelta, date
from functools import partial
import numpy as np
import polars as pl

from okx.store import OrderbookStore
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman, prepare_pillars
from forwards.pchip import PCHIPCurve, reconstruct_forward
from forwards.kalman_ns import NSCarryState, reconstruct_ns_forward

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite",
    batch_days=5
)

First, let's define our date range for testing:


In [16]:
start_date = date(2025, 9, 1)
end_date = date(2025, 9, 4)
dates = [start_date + timedelta(days=i) for i in range((end_date - start_date).days + 1)]

print(f"Testing with dates: {dates[0]} to {dates[-1]}")

Testing with dates: 2025-09-01 to 2025-09-04


## 1. Pillar Preparation

The `prepare_pillars` function fetches and concatenates SWAP (perpetual) and FUTURES orderbook data, creating snapshots with pillars sorted by time-to-maturity (T). Each snapshot contains the SWAP at T=0 followed by FUTURES in ascending maturity order.

Key features:
- Automatic timestamp alignment using asof joins
- Early-roll filtering for near-expiry contracts
- Optional binning for time aggregation
- Returns dict mapping timeMs → pillars_df (already in log-space)


In [17]:
# Prepare pillars with 5-minute binning
snapshots = prepare_pillars(
    store=store,
    inst_family='BTC-USD',
    dates=dates,
    binning='5m',
    min_time_to_expiry_hours=2.0
)

print(f"Number of snapshots: {len(snapshots)}")
print(f"\nFirst snapshot timestamp: {list(snapshots.keys())[0]}")
print(f"\nFirst snapshot shape: {snapshots[list(snapshots.keys())[0]].shape}")
print(f"\nColumns: {snapshots[list(snapshots.keys())[0]].columns}")
print(f"\nFirst few pillars (T, symbol, ln_bid_1_px, ln_ask_1_px):")
first_snapshot = snapshots[list(snapshots.keys())[0]]
print(first_snapshot.select(['T', 'symbol', 'ln_bid_1_px', 'ln_ask_1_px']).head(5))


Number of snapshots: 1151

First snapshot timestamp: 1756685100000

First snapshot shape: (8, 8)

Columns: ['timeMs', 'symbol', 'rel_spread', 'expiry', 'T', 'ln_bid_1_px', 'ln_ask_1_px', 'ln_spread']

First few pillars (T, symbol, ln_bid_1_px, ln_ask_1_px):
shape: (5, 4)
┌──────────┬───────────────────┬─────────────┬─────────────┐
│ T        ┆ symbol            ┆ ln_bid_1_px ┆ ln_ask_1_px │
│ ---      ┆ ---               ┆ ---         ┆ ---         │
│ f64      ┆ str               ┆ f64         ┆ f64         │
╞══════════╪═══════════════════╪═════════════╪═════════════╡
│ 0.0      ┆ BTC-USD-SWAP.OK   ┆ 11.590376   ┆ 11.590377   │
│ 0.011863 ┆ BTC-USD-250905.OK ┆ 11.591237   ┆ 11.591238   │
│ 0.031041 ┆ BTC-USD-250912.OK ┆ 11.592124   ┆ 11.592237   │
│ 0.069397 ┆ BTC-USD-250926.OK ┆ 11.594335   ┆ 11.594336   │
│ 0.165287 ┆ BTC-USD-251031.OK ┆ 11.601465   ┆ 11.601712   │
└──────────┴───────────────────┴─────────────┴─────────────┘


## 2. PCHIP Interpolation with EWMA Smoothing

The `build_forwards_pchip` recipe uses Piecewise Cubic Hermite Interpolating Polynomial (PCHIP) to create smooth forward curves between pillar points.

Key features:
- Shape-preserving interpolation (no overshoots)
- Time-aware EWMA smoothing with α(Δt) = exp(-Δt/τ)
- Frame-rate invariant across different binning intervals
- Works directly with log-space prices from prepare_pillars

The time constant τ (tau_ewma_minutes) controls smoothing strength:
- Smaller τ → more responsive, less smooth
- Larger τ → less responsive, more smooth
- Half-life = τ × ln(2) ≈ 0.693τ

### Direct usage:


In [18]:
# Build PCHIP forward curves with 5-minute time constant
lf_pchip = build_forwards_pchip(
    store=store,
    dates=dates,
    inst_family='BTC-USD',
    binning='5m',
    tau_ewma_minutes=5.0,
    min_time_to_expiry_hours=2.0
)

# Collect and display results
df_pchip = lf_pchip.collect()
print(f"PCHIP curves shape: {df_pchip.shape}")
print(f"\nColumns: {df_pchip.columns}")
print(f"\nFirst few rows:")
print(df_pchip.head())


PCHIP curves shape: (9208, 7)

Columns: ['timeMs', 'T', 'ln_F_bid', 'ln_F_ask', 'F_bid', 'F_ask', 'symbol']

First few rows:
shape: (5, 7)
┌───────────────┬──────────┬───────────┬───────────┬──────────┬──────────┬───────────────────┐
│ timeMs        ┆ T        ┆ ln_F_bid  ┆ ln_F_ask  ┆ F_bid    ┆ F_ask    ┆ symbol            │
│ ---           ┆ ---      ┆ ---       ┆ ---       ┆ ---      ┆ ---      ┆ ---               │
│ i64           ┆ f64      ┆ f64       ┆ f64       ┆ f64      ┆ f64      ┆ str               │
╞═══════════════╪══════════╪═══════════╪═══════════╪══════════╪══════════╪═══════════════════╡
│ 1756685100000 ┆ 0.0      ┆ 11.590376 ┆ 11.590377 ┆ 108052.9 ┆ 108053.0 ┆ BTC-USD-SWAP.OK   │
│ 1756685100000 ┆ 0.011863 ┆ 11.591237 ┆ 11.591238 ┆ 108146.0 ┆ 108146.1 ┆ BTC-USD-250905.OK │
│ 1756685100000 ┆ 0.031041 ┆ 11.592124 ┆ 11.592237 ┆ 108241.9 ┆ 108254.2 ┆ BTC-USD-250912.OK │
│ 1756685100000 ┆ 0.069397 ┆ 11.594335 ┆ 11.594336 ┆ 108481.5 ┆ 108481.6 ┆ BTC-USD-250926.OK │
│ 1756

### Using with store.get_derived()

For caching and reuse, configure the recipe with `functools.partial` and use `store.get_derived()`:


In [19]:
# Configure the recipe with partial
pchip_recipe = partial(
    build_forwards_pchip,
    binning='5m',
    tau_ewma_minutes=5.0,
    min_time_to_expiry_hours=2.0
)

# Use with get_derived for automatic caching
# Note: store.get_derived() is designed for derived recipes, but for this demo
# we'll just call the recipe directly as shown above. In production, you would:
# lf_pchip_cached = store.get_derived(pchip_recipe, start_date, end_date, cache_name='pchip_5m')

print("Recipe configured successfully!")
print(f"Recipe with tau_ewma={5.0}min, binning=5m is ready for reuse")


Recipe configured successfully!
Recipe with tau_ewma=5.0min, binning=5m is ready for reuse


## 3. Kalman-Filtered Nelson-Siegel Carry Model

The `build_forwards_kalman` recipe uses a time-aware Kalman filter with the Nelson-Siegel parametric form to model forward curves. The Nelson-Siegel model represents the forward curve as:

ln F(T) = β₀ + β₁ × exp(-λT) + β₂ × λT × exp(-λT)

Key features:
- Exact OU discretization for frame-rate invariance
- Spread-based adaptive measurement noise
- Three state variables (β₀, β₁, β₂) with independent mean-reversion
- Works directly with log-space prices from prepare_pillars

Parameters:
- `lambda_ns`: Shape parameter (0.5-2.0/year typical for crypto)
- `tau_minutes`: Time constants [τ₀, τ₁, τ₂] in minutes for each beta
- `sigma_per_sqrt_day`: Process volatilities [σ₀, σ₁, σ₂] per sqrt(day)
- `kappa_spread`: Scale factor for spread-based measurement noise (0.5-1.0 typical)

### Direct usage:


In [ ]:
# Build Kalman-filtered forward curves with default parameters
lf_kalman = build_forwards_kalman(
    store=store,
    dates=dates,
    inst_family='BTC-USD',
    binning='5m',
    lambda_ns=1.0,
    # tau_minutes defaults to [2880, 7200, 14400] (2d, 5d, 10d)
    # sigma_per_sqrt_day defaults to [0.01, 0.01, 0.01]
    kappa_spread=0.5,
    min_time_to_expiry_hours=2.0
)

# Collect and display results
df_kalman = lf_kalman.collect()
print(f"Kalman curves shape: {df_kalman.shape}")
print(f"\nColumns: {df_kalman.columns}")
print(f"\nFirst few rows:")
print(df_kalman.head())


Kalman curves shape: (1151, 7)

Columns: ['timeMs', 'beta0', 'beta1', 'beta2', 'lambda_ns', 'ln_F_ref_bid', 'ln_F_ref_ask']

First few rows:
shape: (5, 7)
┌───────────────┬──────────┬──────────┬──────────┬───────────┬──────────────┬──────────────┐
│ timeMs        ┆ beta0    ┆ beta1    ┆ beta2    ┆ lambda_ns ┆ ln_F_ref_bid ┆ ln_F_ref_ask │
│ ---           ┆ ---      ┆ ---      ┆ ---      ┆ ---       ┆ ---          ┆ ---          │
│ i64           ┆ f64      ┆ f64      ┆ f64      ┆ f64       ┆ f64          ┆ f64          │
╞═══════════════╪══════════╪══════════╪══════════╪═══════════╪══════════════╪══════════════╡
│ 1756685100000 ┆ 0.027463 ┆ 0.040854 ┆ 0.055605 ┆ 1.0       ┆ 11.590376    ┆ 11.590377    │
│ 1756685400000 ┆ 0.02738  ┆ 0.040744 ┆ 0.05564  ┆ 1.0       ┆ 11.589347    ┆ 11.589347    │
│ 1756685700000 ┆ 0.027258 ┆ 0.040803 ┆ 0.055291 ┆ 1.0       ┆ 11.589183    ┆ 11.589183    │
│ 1756686000000 ┆ 0.027373 ┆ 0.040899 ┆ 0.055466 ┆ 1.0       ┆ 11.588942    ┆ 11.588943    │
│ 175668

### Custom parameter configuration:

You can customize the mean-reversion and volatility parameters for each beta:


In [21]:
# Custom parameters: faster mean-reversion, higher volatility
kalman_recipe_custom = partial(
    build_forwards_kalman,
    binning='5m',
    lambda_ns=1.5,
    tau_minutes=np.array([1440.0, 2880.0, 7200.0]),  # 1d, 2d, 5d
    sigma_per_sqrt_day=np.array([0.02, 0.015, 0.01]),  # Higher vol for level
    kappa_spread=0.7,
    min_time_to_expiry_hours=2.0
)

print("Kalman recipe configured with custom parameters:")
print("  λ = 1.5 (tighter curvature)")
print("  τ = [1d, 2d, 5d] (faster mean-reversion)")
print("  σ = [0.02, 0.015, 0.01] per √day (higher level volatility)")
print("  κ_spread = 0.7 (more trust in spreads)")


Kalman recipe configured with custom parameters:
  λ = 1.5 (tighter curvature)
  τ = [1d, 2d, 5d] (faster mean-reversion)
  σ = [0.02, 0.015, 0.01] per √day (higher level volatility)
  κ_spread = 0.7 (more trust in spreads)


## 4. Reconstructing Forwards at Specific Tenors

Both models support reconstructing forward bid/ask prices at arbitrary (unlisted) tenors using:
- **PCHIP**: `reconstruct_forward(curve, T_target)` - interpolates from the curve nodes
- **Kalman-NS**: `reconstruct_ns_forward(state, T_target, use_bid)` - evaluates the parametric model

These functions accept scalar or array inputs for batch evaluation.


### PCHIP: Interpolating to specific tenors

The PCHIP model stores curve nodes. We can extract a curve object and interpolate to any tenor:


In [23]:
# Get the first snapshot from PCHIP results
first_time = df_pchip['timeMs'][0]
df_snapshot = df_pchip.filter(pl.col('timeMs') == first_time)

# Create PCHIPCurve object from the snapshot
curve = PCHIPCurve.from_polars(df_snapshot)

print(f"Curve at time {first_time}")
print(f"Pillar tenors (years): {curve.T_nodes}")
print(f"Pillar symbols: {curve.symbols}")

# Reconstruct forwards at specific unlisted tenors
T_target = np.array([0.1, 0.25, 0.5, 0.75, 1.0])  # 1.2M, 3M, 6M, 9M, 12M
F_bid_interp, F_ask_interp = reconstruct_forward(curve, T_target)

print(f"\nInterpolated forwards at custom tenors:")
for i, T in enumerate(T_target):
    print(f"  T={T:.2f}y: F_bid=${F_bid_interp[i]:.2f}, F_ask=${F_ask_interp[i]:.2f}, spread=${F_ask_interp[i]-F_bid_interp[i]:.2f}")


Curve at time 1756685100000
Pillar tenors (years): [0.         0.01186263 0.03104072 0.06939688 0.16528729 0.31871195
 0.56802702 0.81734209]
Pillar symbols: ['BTC-USD-SWAP.OK', 'BTC-USD-250905.OK', 'BTC-USD-250912.OK', 'BTC-USD-250926.OK', 'BTC-USD-251031.OK', 'BTC-USD-251226.OK', 'BTC-USD-260327.OK', 'BTC-USD-260626.OK']

Interpolated forwards at custom tenors:
  T=0.10y: F_bid=$108712.20, F_ask=$108716.21, spread=$4.02
  T=0.25y: F_bid=$109951.24, F_ask=$109965.90, spread=$14.65
  T=0.50y: F_bid=$112032.31, F_ask=$112030.78, spread=$-1.53
  T=0.75y: F_bid=$114110.87, F_ask=$114113.61, spread=$2.74
  T=1.00y: F_bid=$116170.17, F_ask=$116180.39, spread=$10.22


### Kalman-NS: Evaluating the parametric model

The Kalman model stores state parameters (β₀, β₁, β₂, λ). We can evaluate the Nelson-Siegel curve at any tenor:


In [24]:
# Get the first state from Kalman results
first_kalman_time = df_kalman['timeMs'][0]
df_kalman_snapshot = df_kalman.filter(pl.col('timeMs') == first_kalman_time)

# Create NSCarryState object
state = NSCarryState.from_polars(df_kalman_snapshot)

print(f"State at time {first_kalman_time}")
print(f"  β₀ (level):     {state.beta0:.6f}")
print(f"  β₁ (slope):     {state.beta1:.6f}")
print(f"  β₂ (curvature): {state.beta2:.6f}")
print(f"  λ (shape):      {state.lambda_ns:.6f}")

# Evaluate forwards at specific unlisted tenors
T_target = np.array([0.1, 0.25, 0.5, 0.75, 1.0])  # 1.2M, 3M, 6M, 9M, 12M
F_bid_ns = reconstruct_ns_forward(state, T_target, use_bid=True)
F_ask_ns = reconstruct_ns_forward(state, T_target, use_bid=False)

print(f"\nEvaluated forwards at custom tenors:")
for i, T in enumerate(T_target):
    print(f"  T={T:.2f}y: F_bid=${F_bid_ns[i]:.2f}, F_ask=${F_ask_ns[i]:.2f}, spread=${F_ask_ns[i]-F_bid_ns[i]:.2f}")


State at time 1756685100000
  β₀ (level):     0.027463
  β₁ (slope):     0.040854
  β₂ (curvature): 0.055605
  λ (shape):      1.000000

Evaluated forwards at custom tenors:
  T=0.10y: F_bid=$108801.37, F_ask=$108801.47, spread=$0.10
  T=0.25y: F_bid=$109960.90, F_ask=$109961.01, spread=$0.10
  T=0.50y: F_bid=$111983.40, F_ask=$111983.50, spread=$0.10
  T=0.75y: F_bid=$114107.21, F_ask=$114107.32, spread=$0.11
  T=1.00y: F_bid=$116322.22, F_ask=$116322.33, spread=$0.11


### Batch reconstruction

Both functions support array inputs for efficient batch evaluation:


In [26]:
# Generate a dense grid of tenors for curve evaluation
T_grid = np.linspace(0.01, 2.0, 200)  # 200 points from 0.01 to 2 years

# PCHIP: Reconstruct on grid
F_bid_pchip_grid, F_ask_pchip_grid = reconstruct_forward(curve, T_grid)

# Kalman-NS: Evaluate on grid
F_bid_kalman_grid = reconstruct_ns_forward(state, T_grid, use_bid=True)
F_ask_kalman_grid = reconstruct_ns_forward(state, T_grid, use_bid=False)

print(f"Generated {len(T_grid)} forward prices per curve")
print(f"\nPCHIP mid prices at select tenors:")
print(f"  T=0.25y: ${(F_bid_pchip_grid[49] + F_ask_pchip_grid[49])/2:.2f}")
print(f"  T=0.50y: ${(F_bid_pchip_grid[99] + F_ask_pchip_grid[99])/2:.2f}")
print(f"  T=1.00y: ${(F_bid_pchip_grid[199] + F_ask_pchip_grid[199])/2:.2f}")

print(f"\nKalman-NS mid prices at select tenors:")
print(f"  T=0.25y: ${(F_bid_kalman_grid[49] + F_ask_kalman_grid[49])/2:.2f}")
print(f"  T=0.50y: ${(F_bid_kalman_grid[99] + F_ask_kalman_grid[99])/2:.2f}")
print(f"  T=1.00y: ${(F_bid_kalman_grid[199] + F_ask_kalman_grid[199])/2:.2f}")


Generated 200 forward prices per curve

PCHIP mid prices at select tenors:
  T=0.25y: $112031.54
  T=0.50y: $116175.28
  T=1.00y: $124185.68

Kalman-NS mid prices at select tenors:
  T=0.25y: $111983.45
  T=0.50y: $116322.27
  T=1.00y: $125964.74


## Summary

This notebook demonstrated the forward curve modeling infrastructure from `okx.recipes.forwards`:

### Data Preparation
**`prepare_pillars`** - Fetches and concatenates SWAP and FUTURES orderbook data
- Automatic timestamp alignment using asof joins
- Early-roll filtering for near-expiry contracts
- Optional binning for time aggregation
- Returns snapshots with log-space prices sorted by maturity

### Forward Curve Models

**`build_forwards_pchip`** - PCHIP interpolation with time-aware EWMA smoothing
- Shape-preserving interpolation (no overshoots)
- Frame-rate invariant via α(Δt) = exp(-Δt/τ)
- Good for high-frequency data with smooth interpolation between pillars
- Reconstruction: `reconstruct_forward(curve, T_target)`

**`build_forwards_kalman`** - Kalman-filtered Nelson-Siegel carry model
- Parametric model: ln F(T) = ln F_ref + ∫₀ᵀ [β₀ + β₁·e⁻ᵏᵀ + β₂·T·e⁻ᵏᵀ] du
- Three state factors: level (β₀), slope (β₁), curvature (β₂)
- Exact OU discretization for frame-rate invariance
- Spread-based adaptive measurement noise for robust filtering
- Good for noisy data requiring smooth parametric representation
- Reconstruction: `reconstruct_ns_forward(state, T_target, use_bid)`

### Key Features
- Configure recipes with `functools.partial` for reuse and caching
- Both models support efficient batch reconstruction at arbitrary (unlisted) tenors
- Seamless integration with OrderbookStore infrastructure
- Frame-rate invariant across different binning intervals (1m, 5m, etc.)
- Separate bid/ask curve modeling for spread preservation
